In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import os
import pickle

from torkin import TorKin
from config import Config

from scipy.spatial.transform import Rotation

In [2]:
cfg = Config()
kin = TorKin()

No data file named /home/erhan/catkin_ws/src/erhtor_work/tormain/Resources/larm60.txt
bodypart structure for larm is constructed.
No data file named /home/erhan/catkin_ws/src/erhtor_work/tormain/Resources/rarm60.txt
bodypart structure for rarm is constructed.
bodypart structure for head is constructed.


In [3]:
root_dir = "/home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/"
ds_name = "trajs:72_blocks:3_triangle_v_scarce"
ds_type = "interp_0.85"

In [4]:
data_path = os.path.join(root_dir, f'data/torobo/{ds_name}/train_{ds_type}.npy')
print(data_path)

/home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/data/torobo/trajs:72_blocks:3_triangle_v_scarce/train_interp_0.85.npy


In [5]:
with open(data_path, 'rb') as f:
    data_np = np.load(f)

# data_np = np.load(data_path)
print(data_np.shape)

(61, 299, 35)


# Sample Inspection

In [6]:
print(data_np[0, 11, :])

[ 1.10000000e+01  1.31474233e+00  1.15303148e+00  1.13525614e+00
  1.44941022e+00 -6.38663018e-01  9.38644129e-01  5.91524606e-01
 -1.78511221e-02 -3.92443068e-03 -8.91308725e-03  7.77143212e-03
 -3.90806527e-03 -1.89011501e-02  1.39139268e-03  4.43117333e-01
 -2.42614808e-01  8.65000000e-01  4.74497475e-01 -1.25502525e-01
  8.65000000e-01  3.57385192e-01 -1.56882667e-01  8.65000000e-01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -1.77693587e-02 -3.83580150e-03 -8.91295094e-03  7.48425979e-03
 -3.91306625e-03 -1.89848328e-02  1.31651868e-03]


In [7]:
joint_dim = 7
print(data_np[0, 11, 1:1+joint_dim]) #Joints

[ 1.31474233  1.15303148  1.13525614  1.44941022 -0.63866302  0.93864413
  0.59152461]


In [8]:
print(data_np[0, 11, 8:8+joint_dim]) #Velocs

[-0.01785112 -0.00392443 -0.00891309  0.00777143 -0.00390807 -0.01890115
  0.00139139]


In [9]:
print(data_np[0, 11, 15:15+13]) #Target

[ 0.44311733 -0.24261481  0.865       0.47449747 -0.12550253  0.865
  0.35738519 -0.15688267  0.865       0.          0.          0.
  0.        ]


In [10]:
qtorso = np.array([9.71605396e-06, 6.01925199e-05])
print(qtorso)

[9.71605396e-06 6.01925199e-05]


In [11]:
sample_joint = data_np[0, 11, 1:1+joint_dim]
sample_joint

array([ 1.31474233,  1.15303148,  1.13525614,  1.44941022, -0.63866302,
        0.93864413,  0.59152461])

In [12]:
q9 = np.concatenate((qtorso, sample_joint), axis=0)
print(q9)
print(q9.shape)

[ 9.71605396e-06  6.01925199e-05  1.31474233e+00  1.15303148e+00
  1.13525614e+00  1.44941022e+00 -6.38663018e-01  9.38644129e-01
  5.91524606e-01]
(9,)


In [13]:
tix = 1
p, R = kin.forwardkin(tix, q9)

print(p)
print(p.shape)
print(R)
print(R.shape)

[ 0.4524836  -0.16889493  1.22793331]
(3,)
[[-0.99885948  0.04715428  0.00749698]
 [ 0.04687056  0.99831029 -0.0343469 ]
 [-0.00910391 -0.03395634 -0.99938185]]
(3, 3)


In [14]:
# Create rotation object
rot = Rotation.from_matrix(R)

# Convert to Euler angles
euler = rot.as_euler('xyz', degrees=False)
print(euler)  # [roll, pitch, yaw] in degrees

[-3.10762838  0.00910404  3.09470297]


# Stat

In [15]:
all_eulers = []
all_pos = []

In [16]:
qtorso = np.array([9.71605396e-06, 6.01925199e-05])
print(qtorso)

[9.71605396e-06 6.01925199e-05]


In [17]:
for i_traj in range(data_np.shape[0]):
    for j_step in range(data_np.shape[1]):
        data_step = data_np[i_traj, j_step, :]
        joint_vals = data_step[1 : 1+joint_dim]
        q9 = np.concatenate((qtorso, joint_vals), axis=0)
        p, R = kin.forwardkin(tix, q9)
        euler = Rotation.from_matrix(R).as_euler('xyz', 
                                                 degrees=False)
        all_eulers.append(euler)
        all_pos.append(p)

In [18]:
all_eulers = np.array(all_eulers)
all_pos = np.array(all_pos)

print(all_eulers.shape)
print(all_pos.shape)

(18239, 3)
(18239, 3)


In [19]:
euler_min = all_eulers.min(axis=0)
euler_max = all_eulers.max(axis=0)

print(f"Min: {euler_min}")
print(f"Max: {euler_max}")

Min: [-3.14159223 -0.06948102 -3.14159232]
Max: [3.14159251 0.19336319 3.14159256]


In [20]:
pos_min = all_pos.min(axis=0)
pos_max = all_pos.max(axis=0)

print(f"Min: {pos_min}")
print(f"Max: {pos_max}")

Min: [ 0.35035311 -0.26325649  0.86711777]
Max: [ 0.5212371  -0.07633758  1.30122593]


In [21]:
euler_mean = all_eulers.mean(axis=0)
euler_std = all_eulers.std(axis=0)

print(f"Mean: {euler_mean}")
print(f"Std: {euler_std}")

# Normalize to ~N(0, 1)
def normalize_euler(euler):
    return (euler - euler_mean) / (euler_std + 1e-8)


Mean: [-0.24273071  0.00634201  0.96930292]
Std: [3.10951593 0.03861868 2.96679283]


# Create Dataset

In [22]:
# Second pass: build extended array
# Adding 6 values: current timestep (3 pos + 3 euler_norm)
data_extended = np.zeros((data_np.shape[0], 
                          data_np.shape[1], 35 + 6))
print(data_extended.shape)

(61, 299, 41)


In [23]:
for i_traj in range(data_np.shape[0]):
    for j_step in range(data_np.shape[1]):
        data_step = data_np[i_traj, j_step, :]
        joint_vals = data_step[1 : 1+joint_dim]
        q9 = np.concatenate((qtorso, joint_vals), axis=0)
        p, R = kin.forwardkin(tix, q9)
        euler = Rotation.from_matrix(R).as_euler('xyz', 
                                                 degrees=False)
        euler_norm = normalize_euler(euler)

        # Append: original 35 + current (pos 3 + euler 3)
        data_extended[i_traj, j_step, :35] = data_np[i_traj, j_step, :]
        data_extended[i_traj, j_step, 35:38] = p
        data_extended[i_traj, j_step, 38:41] = euler_norm

In [24]:
print(data_extended[0, 11, :])
print(f"Current pose (pos + euler_norm): {data_extended[0, 11, 35:41]}")

[ 1.10000000e+01  1.31474233e+00  1.15303148e+00  1.13525614e+00
  1.44941022e+00 -6.38663018e-01  9.38644129e-01  5.91524606e-01
 -1.78511221e-02 -3.92443068e-03 -8.91308725e-03  7.77143212e-03
 -3.90806527e-03 -1.89011501e-02  1.39139268e-03  4.43117333e-01
 -2.42614808e-01  8.65000000e-01  4.74497475e-01 -1.25502525e-01
  8.65000000e-01  3.57385192e-01 -1.56882667e-01  8.65000000e-01
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -1.77693587e-02 -3.83580150e-03 -8.91295094e-03  7.48425979e-03
 -3.91306625e-03 -1.89848328e-02  1.31651868e-03  4.52483604e-01
 -1.68894934e-01  1.22793331e+00 -9.21332362e-01  7.15204772e-02
  7.16396513e-01]
Current pose (pos + euler_norm): [ 0.4524836  -0.16889493  1.22793331 -0.92133236  0.07152048  0.71639651]


In [26]:
for i in range(11, 21):
    print(f"{i}th pose (pos + euler_norm): {data_extended[0, i, 35:41]}")

11th pose (pos + euler_norm): [ 0.4524836  -0.16889493  1.22793331 -0.92133236  0.07152048  0.71639651]
12th pose (pos + euler_norm): [ 0.45431507 -0.1680215   1.2210318  -0.92046151  0.13665091  0.7158912 ]
13th pose (pos + euler_norm): [ 0.45602985 -0.16709675  1.21409851 -0.91964131  0.1985923   0.71551191]
14th pose (pos + euler_norm): [ 0.45763106 -0.16611958  1.20713469 -0.91887727  0.25708934  0.71525387]
15th pose (pos + euler_norm): [ 0.45912203 -0.16508934  1.20014143 -0.91817453  0.31190928  0.71511182]
16th pose (pos + euler_norm): [ 0.46050628 -0.16400582  1.19311963 -0.91753783  0.3628434   0.71508005]
17th pose (pos + euler_norm): [ 0.46178757 -0.16286928  1.18607002 -0.91697154  0.40970835  0.71515244]
18th pose (pos + euler_norm): [ 0.46296981 -0.16168045  1.17899312 -0.91647957  0.45234721  0.71532252]
19th pose (pos + euler_norm): [ 0.46405713 -0.16044051  1.17188925 -0.91606537  0.49063027  0.71558348]
20th pose (pos + euler_norm): [ 0.46505382 -0.15915109  1.164758

In [27]:
# Save extended data with current pose
output_path = os.path.join(root_dir, 
                           f'data/torobo/{ds_name}/train_{ds_type}_with_pose.npy')
np.save(output_path, data_extended)
print(f"Saved to: {output_path}")

Saved to: /home/arash/catkin_ws/src/feedback_controller/fbc/neural_network/data/torobo/trajs:72_blocks:3_triangle_v_scarce/train_interp_0.85_with_pose.npy


In [28]:
# Save normalization stats for inference
norm_stats = {'euler_mean': euler_mean, 'euler_std': euler_std}
stats_path = os.path.join(root_dir, f'data/torobo/{ds_name}/euler_norm_stats_{ds_type}.pkl')
with open(stats_path, 'wb') as f:
    pickle.dump(norm_stats, f)

In [29]:
# Load a pickle file
with open(stats_path, "rb") as f:
    data_pkl = pickle.load(f)

print(data_pkl)

{'euler_mean': array([-0.24273071,  0.00634201,  0.96930292]), 'euler_std': array([3.10951593, 0.03861868, 2.96679283])}
